In [6]:
import pandas as pd

In [34]:
path = '../../dataset/amazon.parquet'
df = pd.read_parquet(path)

In [35]:
df = df.sample(n=min(50, df.shape[0]), random_state=42).reset_index(drop=True)

In [12]:
from transformers import pipeline

# Load the end-to-end ABSA pipeline
absa_pipeline = pipeline(
    "token-classification",
    model="yangheng/deberta-v3-base-end2end-absa",
    aggregation_strategy="simple" # Groups sub-tokens into whole words
)
aspects = []
for text in df['review'].to_list():
    # Run inference
    predictions = absa_pipeline(text)

    # Post-process the output to make it readable
    item_aspect = []
    for pred in predictions:
        label = pred["entity_group"]
        if label == "O":
            continue

        # Labels look like 'B-ASP-Positive' or 'I-ASP-Negative'
        sentiment = label.split("-")[-1]

        item_aspect.append({
            "aspect": pred["word"],
            "sentiment": sentiment,
            "score": round(pred["score"], 4)
        })

    aspects.append(item_aspect)

Device set to use cuda:0
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [65]:
df_own = pd.read_csv('data/new_aspects_sent.csv')

In [72]:
item = 2
print(df.loc[item]['review'])
aspects[item]

Great movie, disc was delivered as expected.


[{'aspect': 'movie', 'sentiment': 'Positive', 'score': np.float32(0.7024)}]

In [73]:
df_own[df_own['review_idx'] == item]

,review_idx,cluster,aspect,sentiment,mentions,confidence,prob_pos,prob_neu,prob_neg
3,2,0,disc,positive,1,0.986,0.986,0.012,0.002


In [74]:
aspects

[[{'aspect': 'Price', 'sentiment': 'Neutral', 'score': np.float32(0.7859)},
  {'aspect': 'was', 'sentiment': 'Neutral', 'score': np.float32(0.7164)},
  {'aspect': 'Deli', 'sentiment': 'Neutral', 'score': np.float32(0.4784)},
  {'aspect': 'every', 'sentiment': 'Neutral', 'score': np.float32(0.6554)}],
 [{'aspect': 'series', 'sentiment': 'Positive', 'score': np.float32(0.7082)}],
 [{'aspect': 'movie', 'sentiment': 'Positive', 'score': np.float32(0.7024)}],
 [{'aspect': 'writing', 'sentiment': 'Neutral', 'score': np.float32(0.6935)},
  {'aspect': 'editing', 'sentiment': 'Neutral', 'score': np.float32(0.6812)},
  {'aspect': 'acting', 'sentiment': 'Neutral', 'score': np.float32(0.6853)},
  {'aspect': 'ending', 'sentiment': 'Neutral', 'score': np.float32(0.5841)}],
 [{'aspect': 'picture', 'sentiment': 'Neutral', 'score': np.float32(0.7403)},
  {'aspect': 'sound', 'sentiment': 'Neutral', 'score': np.float32(0.7676)}],
 [{'aspect': 'setting', 'sentiment': 'Neutral', 'score': np.float32(0.4647)

In [71]:
df_own

,review_idx,cluster,aspect,sentiment,mentions,confidence,prob_pos,prob_neu,prob_neg
0,0,0,kid,positive,12,0.933750,0.933750,0.058500,0.007917
1,0,1,oval order,positive,1,0.985000,0.985000,0.013000,0.002000
2,1,0,season,positive,1,0.988000,0.988000,0.010000,0.002000
3,2,0,disc,positive,1,0.986000,0.986000,0.012000,0.002000
4,3,0,great writing,positive,1,0.953000,0.953000,0.038000,0.009000
5,4,0,shipment,negative,14,0.655286,0.027429,0.421429,0.550929
6,5,0,extent,positive,11,0.655545,0.619818,0.370636,0.009364
7,6,0,face muscle,positive,2,0.970000,0.970000,0.027500,0.002500
8,7,0,problem,positive,6,0.903667,0.903667,0.083167,0.013333
9,8,0,price,positive,1,0.915000,0.915000,0.082000,0.003000
